# Thermal comfort indices from a uDALES case

This tutorial runs the thermal-comfort chain of `udcomf` on a gathered uDALES case and
looks at its results. It covers:

1. the preflight check of `namoptions` and the gathered files,
2. running the one-step wrapper `ud_run_thermal_comfort.py`,
3. the exchange file (the inputs of the indices),
4. the index file: maps, time series and validity flags,
5. recomputing the indices with your own comfort parameters.

**This notebook needs a completed and gathered simulation.** No case in the repository
satisfies the comfort preflight, so the cells below do not run on a fresh checkout. Set
`CASE` to your case directory first. The simulation must have saved time-averaged statistics
with the temperature and moisture equations active, the facet energy balance with its output
files, time-dependent shortwave and longwave forcing, and `receptor_heights`; see
`tools/python/udcomf/README.md` for the namoptions settings.

## 1. Setup

Run the notebook with the repository virtual environment (`tools/python/.venv`) in which the
optional comfort dependency is installed: `pip install -e 'tools/python[comfort]'`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from netCDF4 import Dataset

TOOLS = Path('..').resolve().parent          # tools/python when run from examples/thermal_comfort
sys.path.insert(0, str(TOOLS))

CASE = Path('/path/to/experiments/001')      # <- your gathered case directory
HEIGHTS = [1.1]                              # receptor heights (m) saved by the simulation
EXPNR = CASE.name

## 2. Preflight

The preflight reads `namoptions` and reports blockers (the chain will not start) and cautions.
It is the first thing the wrapper does; running it here shows what it checks.

In [ ]:
from udcomf.udcomf_export import check_comfort_preflight

report = check_comfort_preflight(CASE)
print('receptor heights :', report.receptor_heights_m)
print('statistics       : tstatsdump=%g s, tstatsgap=%g s, tstatstart=%g s'
      % (report.schedule.tstatsdump, report.schedule.tstatsgap, report.schedule.tstatstart))
for text in report.cautions:
    print('CAUTION:', text)
for text in report.blockers:
    print('BLOCKER:', text)
print('ready:', report.ready)

## 3. Run the chain

The wrapper performs preflight, optional warm-start merge, receptor-level atmosphere, receptor
radiation, exchange assembly and the indices. The radiation step is the slow one; it is
checkpointed per tile and resumes when rerun. For a real case run it outside the notebook with
`tools/ud_run_thermal_comfort.sh <case> c <system> -- --heights ...` so that it survives a closed
session; here it is called in-process for a small case. The log goes to
`<case>/thermal_comfort.<expnr>.log` and to the cell output.

In [ ]:
from udcomf import ud_run_thermal_comfort

status = ud_run_thermal_comfort.main([
    '--case-dir', str(CASE),
    '--heights', *map(str, HEIGHTS),
    # '--indices', 'mrt,utci',                 # a subset of mrt,pet,utci,wbgt
    # '--statistics-source', 'stats_tkslice',  # if the case saved receptor slices only
    # '--data-source', '/path/to/experiments/000',  # warm-start segments, repeat per segment
    '--overwrite',
])
print('exit status', status)

## 4. The exchange file

`thermal_comfort_inputs.h<height>.<expnr>.nc` holds, per record and grid cell, everything the
indices need: air temperature, pressure, humidity, wind at the receptor and at 10 m, the
thirteen radiation planes and the solar angles at the window midpoint. Each record is the mean
over the `tstatsdump` seconds preceding its time stamp.

In [ ]:
from udcomf.heights import height_tag

tag = height_tag(HEIGHTS[0])
exchange = CASE / f'thermal_comfort_inputs.{tag}.{EXPNR}.nc'
with Dataset(exchange) as ds:
    ds.set_auto_mask(False)
    print('records   :', len(ds.dimensions['time']), 'grid:', len(ds.dimensions['x']), 'x', len(ds.dimensions['y']))
    print('variables :', [name for name in ds.variables if ds[name].dimensions == ('x', 'y', 'time')])
    print('window    :', ds.averaging_window_s, 's; solar geometry:', ds.solar_geometry)
    times = ds['time'][:]
    mask = ds['pedestrian_mask'][:].astype(bool)
    ta = np.where(mask[:, :, None], ds['ta'][:], np.nan) - 273.15

plt.figure(figsize=(5, 3))
plt.plot(times / 3600, np.nanmean(ta, axis=(0, 1)))
plt.xlabel('time since start (h)'); plt.ylabel('mean pedestrian air temperature (°C)')
plt.tight_layout()

## 5. The index file

`thermal_comfort_indices.h<height>.<expnr>.nc` has the same grid and records with `mrt`, `pet`,
`utci`, `wbgt` in degrees Celsius, the globe and natural wet-bulb temperatures, and
`validity_flags`. Missing values mark cells outside the pedestrian mask or outside an index's
validated domain; the flags say why.

In [ ]:
indices = CASE / f'thermal_comfort_indices.{tag}.{EXPNR}.nc'
with Dataset(indices) as ds:
    ds.set_auto_mask(False)
    print(ds.indices, '|', ds.comfort_method)
    print('flag meanings:', ds.validity_flag_meanings)
    print('flag counts  :', ds.validity_counts)
    x, y = ds['x'][:], ds['y'][:]
    record = int(np.argmax(np.nanmean(np.where(ds['mrt'][:] > -1e30, ds['mrt'][:], np.nan), axis=(0, 1))))
    fields = {name: np.where(ds[name][:] > -1e30, ds[name][:], np.nan) for name in ('mrt', 'pet', 'utci', 'wbgt') if name in ds.variables}
    flags = ds['validity_flags'][:]

fig, axes = plt.subplots(1, len(fields), figsize=(4 * len(fields), 3.5), constrained_layout=True)
for ax, (name, values) in zip(np.atleast_1d(axes), fields.items()):
    im = ax.pcolormesh(x, y, values[:, :, record].T, shading='nearest')
    ax.set_title(f'{name} at t = {times[record]/3600:.1f} h (°C)'); ax.set_aspect('equal')
    fig.colorbar(im, ax=ax)

In [ ]:
# time series of the spatial mean and the 5-95 % range of each index
fig, axes = plt.subplots(len(fields), 1, figsize=(6, 2.2 * len(fields)), sharex=True, constrained_layout=True)
for ax, (name, values) in zip(np.atleast_1d(axes), fields.items()):
    flat = values.reshape(-1, values.shape[-1])
    ax.fill_between(times / 3600, np.nanpercentile(flat, 5, axis=0), np.nanpercentile(flat, 95, axis=0), alpha=0.3)
    ax.plot(times / 3600, np.nanmean(flat, axis=0))
    ax.set_ylabel(f'{name} (°C)')
axes[-1].set_xlabel('time since start (h)') if len(fields) > 1 else axes.set_xlabel('time since start (h)')

In [ ]:
# where and why an index is missing: bit 2 UTCI domain, 4 WBGT domain, 8 WBGT nonconverged,
# 16 PET failed, 1 invalid humidity, 32 humidity capped, 64 invalid input
for bit, label in ((1, 'invalid humidity'), (2, 'UTCI outside domain'), (4, 'WBGT outside domain'),
                   (8, 'WBGT nonconverged'), (16, 'PET failed'), (32, 'humidity capped'), (64, 'invalid input')):
    n = int(np.count_nonzero((flags > 0) & (flags & bit)))
    print(f'{label:<20s} {n:8d} cell-records')

## 6. Your own comfort parameters

All person, sensor and domain values of the indices are read from a JSON file. Copy
`comfort_parameters.json` from this directory (it starts as a copy of the packaged defaults),
edit it, and recompute only the indices from the existing exchange file; the radiation step is
not repeated.

In [ ]:
import json
from udcomf import calculate_indices as index_step

parameters = Path('comfort_parameters.json').resolve()
values = json.loads(parameters.read_text())
values['pet_clo'] = 0.5          # lighter clothing, for example
my_parameters = CASE / 'my_comfort_parameters.json'
my_parameters.write_text(json.dumps(values, indent=2))

out = index_step.run([exchange], indices=('mrt', 'pet'), parameters=my_parameters, overwrite=True)
print(out)

The output attribute `comfort_parameters` records the values used, so two index files can
always be told apart. For the meaning of every key see `tools/python/udcomf/README.md`,
section *MRT, PET, UTCI and WBGT*.